# Module 3 — 02: Preprocessing — Heart Disease Datasets

This notebook preprocesses three heart-disease-related datasets that are already available locally under `data/raw/`: cleaning the target variable, handling missing/invalid values, splitting into train/test, encoding categorical features, and scaling numeric features, for each dataset in turn.

All preprocessing logic lives in `src/data/loader.py`, `src/data/preprocessing.py`, and `src/data/split.py` (plus dataset settings in `src/config.py`) — this notebook only *calls* those functions and reports on their results, instead of re-implementing the logic inline. That keeps a single source of truth: the exact same functions are used here and by `src/main.py`'s `process_dataset()`, which is what actually produces the `data/processed/<dataset>/{train,test}.csv` files used for modeling.

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd

from config import (
    DATASET1, DATASET2, DATASET3,
    IQR_MULTIPLIER, FEATURE_VARIANCE_THRESHOLD, FEATURE_CORRELATION_THRESHOLD,
    N_SPLITS,
)
from data.loader import load_raw_dataset, infer_column_types
from data.preprocessing import (
    mark_invalid_zeros_as_missing,
    clean_target,
    compute_class_weights,
    fit_outlier_bounds,
    apply_outlier_bounds,
    fit_preprocessor,
    transform_features,
    fit_feature_selector,
    apply_feature_selector,
)
from data.split import split_train_test, stratified_kfold_splits

pd.set_option('display.max_columns', 20)
print('Modules imported from src/ successfully.')

Modules imported from src/ successfully.


## Dataset 1: Personal Key Indicators of Heart Disease (2022)

### 1. Load Data

In [2]:
df1 = load_raw_dataset(DATASET1.raw_path, target_column=DATASET1.target_column)
print('Loaded file:', DATASET1.raw_path)
print('Shape:', df1.shape)
df1.head()

Loaded file: D:\AIO2026\Module3\Conquer\healthcare-risk-prediction\data\raw\dataset1\heart_2022_with_nans.csv
Shape: (445132, 40)


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,...,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,NaN,No,...,NaN,NaN,NaN,No,No,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
1,Alabama,Female,Excellent,0.0,0.0,NaN,No,6.0,NaN,No,...,1.60,68.04,26.57,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,No
2,Alabama,Female,Very good,2.0,3.0,Within past year (anytime less than 12 months ...,Yes,5.0,NaN,No,...,1.57,63.50,25.61,No,No,No,No,NaN,No,Yes
3,Alabama,Female,Excellent,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,NaN,No,...,1.65,63.50,23.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
4,Alabama,Female,Fair,2.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,NaN,No,...,1.57,53.98,21.77,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,No


**Result:** Loaded via `load_raw_dataset()`, which reads the CSV, strips column-name whitespace, and drops any fully-empty columns — no dataset-specific logic here, the same function is reused for all three datasets. We load `heart_2022_with_nans.csv` on purpose (rather than the pre-cleaned `_no_nans` variant), since it lets us demonstrate a realistic preprocessing workflow that includes handling missing values. The loaded dataframe has **445,132 rows and 40 columns**, matching the official 2022 BRFSS update of this dataset.

### 2. Clean Target + Class Weights

In [3]:
df1 = clean_target(df1, target_column=DATASET1.target_column)
print('Shape after dropping rows with missing target:', df1.shape)
print(df1[DATASET1.target_column].value_counts())

Shape after dropping rows with missing target: (442067, 40)
HadHeartAttack
0    416959
1     25108
Name: count, dtype: int64


**Result:** `clean_target()` drops the 3,065 rows (0.69%) with a missing target and maps `HadHeartAttack` from Yes/No strings to a binary 0/1 integer, leaving **442,067 usable rows** (416,959 negatives / 25,108 positives).

In [4]:
class_weights1 = compute_class_weights(df1[DATASET1.target_column])
print('Class weights (balanced, informational only):', class_weights1)

Class weights (balanced, informational only): {0: 0.5301084758933132, 1: 8.803309702086985}


**Result:** `compute_class_weights()` reports balanced class weights of approximately **{0: 0.53, 1: 8.80}** — a positive example (`HadHeartAttack = 1`) is weighted about **16.6x** more than a negative one, reflecting the ~17:1 class imbalance above. This project's chosen class-imbalance policy (see `docs/qa-scope-methodology-review-handoff.md`, finding F1) is to pass weights like these as `sample_weight` at model-fit time (`src/experiments/run_models.py`, driven by `config.yaml`'s `balance_training` flag) — **not** to resample rows here in preprocessing, since doing both would double-correct for the same imbalance.

### 3. Column Types + Train/Test Split

In [5]:
numeric_cols1, categorical_cols1 = infer_column_types(
    df1, DATASET1.numeric_columns, target_column=DATASET1.target_column
)
print(f'Numeric columns ({len(numeric_cols1)}):', numeric_cols1)
print(f'Categorical columns ({len(categorical_cols1)}):', categorical_cols1)

Numeric columns (6): ['PhysicalHealthDays', 'MentalHealthDays', 'SleepHours', 'HeightInMeters', 'WeightInKilograms', 'BMI']
Categorical columns (33): ['State', 'Sex', 'GeneralHealth', 'LastCheckupTime', 'PhysicalActivities', 'RemovedTeeth', 'HadAngina', 'HadStroke', 'HadAsthma', 'HadSkinCancer', 'HadCOPD', 'HadDepressiveDisorder', 'HadKidneyDisease', 'HadArthritis', 'HadDiabetes', 'DeafOrHardOfHearing', 'BlindOrVisionDifficulty', 'DifficultyConcentrating', 'DifficultyWalking', 'DifficultyDressingBathing', 'DifficultyErrands', 'SmokerStatus', 'ECigaretteUsage', 'ChestScan', 'RaceEthnicityCategory', 'AgeCategory', 'AlcoholDrinkers', 'HIVTesting', 'FluVaxLast12', 'PneumoVaxEver', 'TetanusLast10Tdap', 'HighRiskLastYear', 'CovidPos']


**Result:** `infer_column_types()` splits the 39 feature columns using `DatasetConfig.numeric_columns` as the explicit allow-list: the 6 listed columns are numeric, and the remaining 33 are treated as categorical.

In [6]:
split1 = split_train_test(df1, target_column=DATASET1.target_column)
train1, test1 = split1.train_df, split1.test_df
print('Train rows:', len(train1), '| Test rows:', len(test1))

Train rows: 353653 | Test rows: 88414


**Result:** `split_train_test()` performs a stratified 80/20 split **before** any imputer/encoder/scaler is fitted, so that all subsequent fitting happens on `train1` only and nothing about `test1` leaks into it. This gives **353,653 training rows** and **88,414 test rows**, with the ~5.68% positive rate preserved in both splits via stratification. Dataset 1 has no `iqr_outlier_columns` configured (its features are mostly binary/discrete survey answers, not continuous clinical measurements — see `docs/qa-scope-methodology-review-handoff.md`, finding F5), so outlier clipping is skipped for this dataset.

### 4. Fit Preprocessor (Impute / Encode / Scale)

In [7]:
fitted1, train1_processed = fit_preprocessor(
    train1, numeric_cols1, categorical_cols1, target_column=DATASET1.target_column
)
test1_processed = transform_features(test1, fitted1, target_column=DATASET1.target_column)
print('Processed train shape:', train1_processed.shape)
print('Processed test shape:', test1_processed.shape)
train1_processed.head()

Processed train shape: (353653, 133)
Processed test shape: (88414, 133)


,HadHeartAttack,PhysicalHealthDays,MentalHealthDays,SleepHours,HeightInMeters,WeightInKilograms,BMI,Sex,PhysicalActivities,HadAngina,...,AgeCategory_Age 70 to 74,AgeCategory_Age 75 to 79,AgeCategory_Age 80 or older,"TetanusLast10Tdap_No, did not receive any tetanus shot in the past 10 years","TetanusLast10Tdap_Yes, received Tdap","TetanusLast10Tdap_Yes, received tetanus shot but not sure what type","TetanusLast10Tdap_Yes, received tetanus shot, but not Tdap",CovidPos_No,CovidPos_Tested positive using home test without a health professional,CovidPos_Yes
0,0,1.254304,3.089113,-2.031888,-0.217177,-1.679847,-1.796724,0,1,0,...,0,0,0,1,0,0,0,1,0,0
1,0,-0.491921,-0.155749,-0.015529,1.422739,1.493389,0.735506,1,0,0,...,0,0,0,0,0,1,0,1,0,0
2,0,3.000530,-0.516289,-0.015529,-0.506574,-1.191657,-1.119743,0,0,0,...,0,1,0,0,0,0,1,1,0,0
3,0,-0.491921,-0.516289,0.656591,-0.024246,-0.104235,-0.157464,1,1,0,...,0,0,0,1,0,0,0,1,0,0
4,0,0.090154,-0.516289,0.656591,-1.278299,-1.612831,-1.337344,0,0,0,...,0,1,0,1,0,0,0,1,0,0


**Result:** `fit_preprocessor()` fits a median imputer (numeric columns) and mode imputer (categorical columns), label-encodes the 22 binary categorical columns to 0/1, one-hot encodes the 11 multi-category columns, and standardizes the 6 numeric columns to zero mean/unit variance — all fit on `train1` only. `transform_features()` then applies those exact fitted objects to `test1`, filling in any category seen in test but not train with the training-time most-frequent value. Both splits end up with the same **132 feature columns + target** (after the one-hot expansion from 39 to 132), matching row counts of 353,653 and 88,414 respectively.

### 5. Feature Reduction

In [8]:
selector1 = fit_feature_selector(
    train1_processed,
    target_column=DATASET1.target_column,
    variance_threshold=FEATURE_VARIANCE_THRESHOLD,
    correlation_threshold=FEATURE_CORRELATION_THRESHOLD,
)
print('Low-variance columns dropped:', selector1.dropped_low_variance)
print('Highly-correlated columns dropped:', selector1.dropped_correlated)

train1_final = apply_feature_selector(train1_processed, selector1, target_column=DATASET1.target_column)
test1_final = apply_feature_selector(test1_processed, selector1, target_column=DATASET1.target_column)
print('Final train shape:', train1_final.shape)
print('Final test shape:', test1_final.shape)

Low-variance columns dropped: []
Highly-correlated columns dropped: ['CovidPos_Yes']
Final train shape: (353653, 132)
Final test shape: (88414, 132)


**Result:** `fit_feature_selector()` (fit on `train1_processed` only, then applied identically to both splits via `apply_feature_selector()`) drops no low-variance columns, but drops **`CovidPos_Yes`** for being highly correlated (r ≈ 0.93) with `CovidPos_No` — expected, since the raw `CovidPos` column has 3 categories (`No`: 270,055, `Yes`: 110,877, `Tested positive using home test...`: 13,436) where the small third category makes `Yes` and `No` near-perfect complements. Both splits shrink from 132 to **131 feature columns** (132 including the target) with no real loss of information.

### 6. Stratified K-Fold Validation

Besides the held-out test split (never touched again after Section 3), the *training* split itself is further divided for model selection: `stratified_kfold_splits()` partitions `train1_final` into `N_SPLITS=5` stratified folds -- each fold takes a turn as the validation set (1/5 of the training rows) while the other 4 folds serve as that round's training data (4/5), rotating so every row is validated exactly once. Stratification keeps the target's class proportions consistent across folds, the same way the original train/test split does. This mirrors exactly what `src/main.py` computes and persists to `DATASET1.kfold_indices_path` (`kfold_indices.csv`, one `fold` label per row, aligned row-for-row with `DATASET1.train_csv_path`).

In [9]:
folds = stratified_kfold_splits(train1_final, target_column=DATASET1.target_column, n_splits=N_SPLITS)
print(f'Number of folds: {len(folds)}')

fold_rows = []
for fold_number, (train_idx, val_idx) in enumerate(folds):
    val_target = train1_final[DATASET1.target_column].iloc[val_idx]
    fold_rows.append({
        'fold': fold_number,
        'train_rows': len(train_idx),
        'val_rows': len(val_idx),
        'val_positive_rate_%': round(val_target.mean() * 100, 2),
    })
fold_summary = pd.DataFrame(fold_rows)
print(f'Overall positive rate: {(train1_final[DATASET1.target_column].mean() * 100):.2f}%')
fold_summary

Number of folds: 5
Overall positive rate: 5.68%


,fold,train_rows,val_rows,val_positive_rate_%
0,0,282922,70731,5.68
1,1,282922,70731,5.68
2,2,282922,70731,5.68
3,3,282923,70730,5.68
4,4,282923,70730,5.68


**Result:** All 5 folds get essentially equal train (~4/5) / validation (~1/5) sizes, and each fold's validation positive rate stays close to the overall positive rate above -- confirming `stratified_kfold_splits()` preserves the class balance across folds, not just across the original train/test split. `src/main.py` runs this same call and writes the resulting fold labels to `DATASET1.kfold_indices_path`.

## Dataset 2: Heart Failure Prediction

### 1. Load Data

In [10]:
df2 = load_raw_dataset(DATASET2.raw_path, target_column=DATASET2.target_column)
print('Loaded file:', DATASET2.raw_path)
print('Shape:', df2.shape)
df2.head()

Loaded file: D:\AIO2026\Module3\Conquer\healthcare-risk-prediction\data\raw\dataset2\heart.csv
Shape: (918, 12)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


**Result:** Loaded via the same `load_raw_dataset()` used for Dataset 1. **918 rows and 12 columns**, target `HeartDisease` already encoded as 0/1.

### 2. Zero-as-Missing + Clean Target + Class Weights

In [11]:
df2 = mark_invalid_zeros_as_missing(df2, DATASET2.zero_as_missing_columns)
print(df2[DATASET2.zero_as_missing_columns].isna().sum())

Cholesterol    172
RestingBP        1
dtype: int64


**Result:** `mark_invalid_zeros_as_missing()` converts `Cholesterol`/`RestingBP` zero-placeholder values to genuine `NaN` (172 and 1 rows respectively) — a `0` reading for either is not physiologically possible, so treating it as "not measured" lets the normal median imputation below handle it correctly instead of the model seeing an impossible zero value.

In [12]:
df2 = clean_target(df2, target_column=DATASET2.target_column)
print('Shape after dropping rows with missing target:', df2.shape)
print(df2[DATASET2.target_column].value_counts())

Shape after dropping rows with missing target: (918, 12)
HeartDisease
1    508
0    410
Name: count, dtype: int64


**Result:** `HeartDisease` was already a clean numeric 0/1 column with no missing values, so `clean_target()` only confirms/casts the dtype here — the row count stays at 918 (508 positive / 410 negative, 55.34% / 44.66%).

In [13]:
class_weights2 = compute_class_weights(df2[DATASET2.target_column])
print('Class weights (balanced, informational only):', class_weights2)

Class weights (balanced, informational only): {0: 1.1195121951219513, 1: 0.9035433070866141}


**Result:** With the class split close to balanced, `compute_class_weights()` reports approximately **{0: 1.12, 1: 0.90}** — barely different from 1.0, confirming this dataset needs very little correction for imbalance, unlike Dataset 1 or Dataset 3.

### 3. Column Types + Train/Test Split

In [14]:
numeric_cols2, categorical_cols2 = infer_column_types(
    df2, DATASET2.numeric_columns, target_column=DATASET2.target_column
)
print(f'Numeric columns ({len(numeric_cols2)}):', numeric_cols2)
print(f'Categorical columns ({len(categorical_cols2)}):', categorical_cols2)

split2 = split_train_test(df2, target_column=DATASET2.target_column)
train2, test2 = split2.train_df, split2.test_df
print('Train rows:', len(train2), '| Test rows:', len(test2))

Numeric columns (6): ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
Categorical columns (5): ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
Train rows: 734 | Test rows: 184


**Result:** 6 numeric columns (`Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`, `Oldpeak`) and 5 categorical columns. The stratified 80/20 split gives **734 training rows** and **184 test rows**.

### 4. Outlier Handling (IQR / Tukey Fences)

Dataset 2 is the only one of the three datasets configured with `iqr_outlier_columns` (see `docs/qa-scope-methodology-review-handoff.md`, finding F5): its numeric columns are genuine continuous clinical measurements, unlike the mostly-binary/discrete survey data in Dataset 1 and Dataset 3, where a real high value (e.g. a genuinely high BMI) is disease signal, not noise. `fit_outlier_bounds()` is fit on **`train2` only**, then the same bounds are applied to both `train2` and `test2` via `apply_outlier_bounds()` — this must happen *after* the split (never on the full dataset at once), otherwise information about the test split's own extreme values would leak into the bounds used to clip it. `FastingBS` is deliberately excluded from `DATASET2.iqr_outlier_columns` even though it's numeric-typed: it's a 0/1 flag, and an IQR fence on a mostly-0 column would collapse to a single point and clip away the entire minority (`FastingBS=1`) signal.

In [15]:
outlier_bounds2 = fit_outlier_bounds(train2, DATASET2.iqr_outlier_columns, multiplier=IQR_MULTIPLIER)
for col, (lower, upper) in outlier_bounds2.bounds.items():
    n_clipped_train = ((train2[col] < lower) | (train2[col] > upper)).sum()
    print(f'{col}: bounds=({lower:.2f}, {upper:.2f}), {n_clipped_train} value(s) in train would be clipped')

train2 = apply_outlier_bounds(train2, outlier_bounds2)
test2 = apply_outlier_bounds(test2, outlier_bounds2)
print('Applied IQR outlier clipping to:', DATASET2.iqr_outlier_columns)

Age: bounds=(26.00, 82.00), 0 value(s) in train would be clipped
RestingBP: bounds=(88.50, 172.50), 20 value(s) in train would be clipped
Cholesterol: bounds=(109.50, 377.50), 20 value(s) in train would be clipped
MaxHR: bounds=(63.50, 211.50), 2 value(s) in train would be clipped
Oldpeak: bounds=(-2.25, 3.75), 13 value(s) in train would be clipped
Applied IQR outlier clipping to: ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']


**Result:** Bounds are fit on the 734-row training split only. `Age` has no values outside its Tukey fence. `RestingBP`, `Cholesterol` (computed after the zero-as-missing step above), `MaxHR`, and `Oldpeak` each flag a handful of potential outliers in the training split, consistent with the boxplots and IQR bounds shown in `01_eda.ipynb` (computed there on the full dataset for exploratory purposes only — here the bounds come from `train2` alone, which is the leakage-safe version actually used for preprocessing). Values are clipped in place, never dropped, so row counts are unchanged.

### 5. Fit Preprocessor (Impute / Encode / Scale)

In [16]:
fitted2, train2_processed = fit_preprocessor(
    train2, numeric_cols2, categorical_cols2, target_column=DATASET2.target_column
)
test2_processed = transform_features(test2, fitted2, target_column=DATASET2.target_column)
print('Processed train shape:', train2_processed.shape)
print('Processed test shape:', test2_processed.shape)
train2_processed.head()

Processed train shape: (734, 19)
Processed test shape: (184, 19)


,HeartDisease,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex,ExerciseAngina,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,1,0.970012,0.375636,-0.553445,1.835497,-0.324929,0.333747,1,1,0,1,0,0,0,0,1,0,1,0
1,0,0.122028,-1.313376,-0.615981,1.835497,1.690655,-0.445018,1,0,0,1,0,0,0,0,1,0,0,1
2,1,0.546020,-0.148540,1.968809,1.835497,-0.247406,0.625784,0,1,1,0,0,0,0,0,1,0,1,0
3,1,-0.725956,1.598714,-0.115699,-0.544812,-0.479974,-0.834400,1,1,1,0,0,0,0,1,0,0,1,0
4,1,-0.407962,0.725087,-0.115699,1.835497,0.101445,-0.152981,1,1,1,0,0,0,0,1,0,0,1,0


**Result:** Same `fit_preprocessor()`/`transform_features()` pair as Dataset 1, fit on `train2` only: median imputation for the (now-NaN-containing) numeric columns, mode imputation for categorical columns, label encoding for the 2 binary columns (`Sex`, `ExerciseAngina`), one-hot encoding for the 3 multi-category columns (`ChestPainType`, `RestingECG`, `ST_Slope`), and standardization of the 6 numeric columns. Both splits end up with **18 feature columns + target** (19 total columns).

### 6. Feature Reduction

In [17]:
selector2 = fit_feature_selector(
    train2_processed,
    target_column=DATASET2.target_column,
    variance_threshold=FEATURE_VARIANCE_THRESHOLD,
    correlation_threshold=FEATURE_CORRELATION_THRESHOLD,
)
print('Low-variance columns dropped:', selector2.dropped_low_variance)
print('Highly-correlated columns dropped:', selector2.dropped_correlated)

train2_final = apply_feature_selector(train2_processed, selector2, target_column=DATASET2.target_column)
test2_final = apply_feature_selector(test2_processed, selector2, target_column=DATASET2.target_column)
print('Final train shape:', train2_final.shape)
print('Final test shape:', test2_final.shape)

Low-variance columns dropped: []
Highly-correlated columns dropped: []
Final train shape: (734, 19)
Final test shape: (184, 19)


**Result:** Nothing gets dropped — Dataset 2 has no near-constant columns and no feature pair exceeds the 0.9 correlation threshold on the training split. Both splits stay at **18 feature columns + target** (19 total columns). This is a useful negative result: feature reduction is applied uniformly to all 3 datasets, but only actually removes something where the (training) data calls for it.

### 7. Stratified K-Fold Validation

Besides the held-out test split (never touched again after Section 3), the *training* split itself is further divided for model selection: `stratified_kfold_splits()` partitions `train2_final` into `N_SPLITS=5` stratified folds -- each fold takes a turn as the validation set (1/5 of the training rows) while the other 4 folds serve as that round's training data (4/5), rotating so every row is validated exactly once. Stratification keeps the target's class proportions consistent across folds, the same way the original train/test split does. This mirrors exactly what `src/main.py` computes and persists to `DATASET2.kfold_indices_path` (`kfold_indices.csv`, one `fold` label per row, aligned row-for-row with `DATASET2.train_csv_path`).

In [18]:
folds = stratified_kfold_splits(train2_final, target_column=DATASET2.target_column, n_splits=N_SPLITS)
print(f'Number of folds: {len(folds)}')

fold_rows = []
for fold_number, (train_idx, val_idx) in enumerate(folds):
    val_target = train2_final[DATASET2.target_column].iloc[val_idx]
    fold_rows.append({
        'fold': fold_number,
        'train_rows': len(train_idx),
        'val_rows': len(val_idx),
        'val_positive_rate_%': round(val_target.mean() * 100, 2),
    })
fold_summary = pd.DataFrame(fold_rows)
print(f'Overall positive rate: {(train2_final[DATASET2.target_column].mean() * 100):.2f}%')
fold_summary

Number of folds: 5
Overall positive rate: 55.31%


,fold,train_rows,val_rows,val_positive_rate_%
0,0,587,147,55.78
1,1,587,147,55.10
2,2,587,147,55.10
3,3,587,147,55.10
4,4,588,146,55.48


**Result:** All 5 folds get essentially equal train (~4/5) / validation (~1/5) sizes, and each fold's validation positive rate stays close to the overall positive rate above -- confirming `stratified_kfold_splits()` preserves the class balance across folds, not just across the original train/test split. `src/main.py` runs this same call and writes the resulting fold labels to `DATASET2.kfold_indices_path`.

## Dataset 3: Heart Disease Health Indicators (BRFSS 2015)

### 1. Load Data

In [19]:
df3 = load_raw_dataset(DATASET3.raw_path, target_column=DATASET3.target_column)
print('Loaded file:', DATASET3.raw_path)
print('Shape:', df3.shape)
df3.head()

Loaded file: D:\AIO2026\Module3\Conquer\healthcare-risk-prediction\data\raw\dataset3\heart_disease_health_indicators_BRFSS2015.csv
Shape: (253680, 22)


,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


**Result:** Loaded via `load_raw_dataset()`. **253,680 rows and 22 columns**, all already numerically encoded, with the binary target `HeartDiseaseorAttack`.

### 2. Drop Duplicates + Clean Target + Class Weights

In [20]:
before_dup = len(df3)
if DATASET3.drop_duplicate_rows:
    df3 = df3.drop_duplicates().reset_index(drop=True)
print(f'Rows before dropping duplicates: {before_dup}')
print(f'Rows after dropping duplicates: {len(df3)}')
print(f'Duplicate rows dropped: {before_dup - len(df3)}')

Rows before dropping duplicates: 253680
Rows after dropping duplicates: 229781
Duplicate rows dropped: 23899


**Result:** `DATASET3.drop_duplicate_rows=True` (unlike Dataset 1/2), so `src/main.py`'s pipeline drops the **23,899 exact-duplicate rows (9.42%)** here before target-cleaning — this mirrors the EDA notebook's finding that, with 22 binary/small-discrete-scale features, distinct respondents can easily share an identical answer pattern. This leaves 229,781 rows.

In [21]:
df3 = clean_target(df3, target_column=DATASET3.target_column)
print('Shape after dropping rows with missing target:', df3.shape)
print(df3[DATASET3.target_column].value_counts())

Shape after dropping rows with missing target: (229781, 22)
HeartDiseaseorAttack
0    206064
1     23717
Name: count, dtype: int64


**Result:** `HeartDiseaseorAttack` had no missing values, so `clean_target()` only casts it to `int` here — 229,781 rows before and after, now split 206,064 negatives (89.68%) / 23,717 positives (10.32%): slightly less imbalanced than the raw 90.58%/9.42% split, since a disproportionate share of the dropped duplicate rows were negative.

In [22]:
class_weights3 = compute_class_weights(df3[DATASET3.target_column])
print('Class weights (balanced, informational only):', class_weights3)

Class weights (balanced, informational only): {0: 0.5575476550974454, 1: 4.844225660918329}


**Result:** With 89.68% negatives vs. 10.32% positives (after duplicate removal), `compute_class_weights()` reports approximately **{0: 0.56, 1: 4.84}** — a positive example is weighted about **8.7x** more than a negative one, less extreme than Dataset 1's ~16.6x since Dataset 3's positive rate is roughly double Dataset 1's. (Note: `01_eda.ipynb` reports a slightly different ~9.6x ratio for Dataset 3 — that notebook computes weights on the *raw*, non-deduplicated data, since deduplication is a preprocessing step that notebook doesn't apply; the two numbers legitimately differ for that reason.)

### 3. Column Types + Train/Test Split

In [23]:
numeric_cols3, categorical_cols3 = infer_column_types(
    df3, DATASET3.numeric_columns, target_column=DATASET3.target_column
)
print(f'Numeric columns ({len(numeric_cols3)}):', numeric_cols3)
print(f'Categorical columns ({len(categorical_cols3)}):', categorical_cols3)

split3 = split_train_test(df3, target_column=DATASET3.target_column)
train3, test3 = split3.train_df, split3.test_df
print('Train rows:', len(train3), '| Test rows:', len(test3))

Numeric columns (7): ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']
Categorical columns (14): ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']
Train rows: 183824 | Test rows: 45957


**Result:** 7 numeric/ordinal columns and 14 categorical columns. The stratified 80/20 split gives **183,824 training rows** and **45,957 test rows**. Dataset 3 has no `iqr_outlier_columns` configured (like Dataset 1, its features are mostly binary/discrete survey answers), so outlier clipping is skipped here too.

### 4. Fit Preprocessor (Impute / Encode / Scale)

In [24]:
fitted3, train3_processed = fit_preprocessor(
    train3, numeric_cols3, categorical_cols3, target_column=DATASET3.target_column
)
test3_processed = transform_features(test3, fitted3, target_column=DATASET3.target_column)
print('Processed train shape:', train3_processed.shape)
print('Processed test shape:', test3_processed.shape)
train3_processed.head()

Processed train shape: (183824, 24)
Processed test shape: (45957, 24)


,HeartDiseaseorAttack,BMI,GenHlth,MentHlth,PhysHlth,Age,Education,Income,HighBP,HighChol,...,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,DiffWalk,Sex,Diabetes_0.0,Diabetes_1.0,Diabetes_2.0
0,0,-0.249055,2.251163,3.438176,2.796650,0.617706,-0.988815,-2.337071,1,0,...,0,0,0,1,0,1,0,1,0,0
1,0,-1.426816,-0.565182,-0.454350,-0.517075,0.940686,0.019086,1.008672,0,0,...,1,1,0,1,0,0,0,1,0,0
2,0,2.400908,1.312382,-0.065098,0.035213,-0.028253,-0.988815,0.052746,1,0,...,1,1,0,1,0,1,0,1,0,0
3,0,-0.396275,-1.503964,-0.454350,-0.517075,0.617706,-0.988815,1.008672,1,1,...,1,1,0,1,0,0,1,1,0,0
4,1,0.487046,0.373600,-0.454350,2.796650,1.263665,0.019086,-1.381144,1,1,...,1,1,0,1,0,1,0,0,0,1


**Result:** Same `fit_preprocessor()`/`transform_features()` pair, fit on `train3` only: median imputation (a no-op here, since Dataset 3 has no missing values to begin with), mode imputation, label encoding for the 13 binary flags, one-hot encoding for the single 3-category `Diabetes` column, and standardization of the 7 numeric/ordinal columns. Both splits end up with **23 feature columns + target** (24 total columns).

### 5. Feature Reduction

In [25]:
selector3 = fit_feature_selector(
    train3_processed,
    target_column=DATASET3.target_column,
    variance_threshold=FEATURE_VARIANCE_THRESHOLD,
    correlation_threshold=FEATURE_CORRELATION_THRESHOLD,
)
print('Low-variance columns dropped:', selector3.dropped_low_variance)
print('Highly-correlated columns dropped:', selector3.dropped_correlated)

train3_final = apply_feature_selector(train3_processed, selector3, target_column=DATASET3.target_column)
test3_final = apply_feature_selector(test3_processed, selector3, target_column=DATASET3.target_column)
print('Final train shape:', train3_final.shape)
print('Final test shape:', test3_final.shape)

Low-variance columns dropped: []
Highly-correlated columns dropped: ['Diabetes_2.0']
Final train shape: (183824, 23)
Final test shape: (45957, 23)


**Result:** `fit_feature_selector()` (fit on the 183,824-row training split) drops no low-variance columns, but drops **`Diabetes_2.0`** for being highly correlated with `Diabetes_0.0` — the same dominant-plus-rare-category pattern as Dataset 1's `CovidPos`. Both splits shrink from 23 to **22 feature columns** (24 total columns including the target before the drop, 23 after).

### 6. Stratified K-Fold Validation

Besides the held-out test split (never touched again after Section 3), the *training* split itself is further divided for model selection: `stratified_kfold_splits()` partitions `train3_final` into `N_SPLITS=5` stratified folds -- each fold takes a turn as the validation set (1/5 of the training rows) while the other 4 folds serve as that round's training data (4/5), rotating so every row is validated exactly once. Stratification keeps the target's class proportions consistent across folds, the same way the original train/test split does. This mirrors exactly what `src/main.py` computes and persists to `DATASET3.kfold_indices_path` (`kfold_indices.csv`, one `fold` label per row, aligned row-for-row with `DATASET3.train_csv_path`).

In [26]:
folds = stratified_kfold_splits(train3_final, target_column=DATASET3.target_column, n_splits=N_SPLITS)
print(f'Number of folds: {len(folds)}')

fold_rows = []
for fold_number, (train_idx, val_idx) in enumerate(folds):
    val_target = train3_final[DATASET3.target_column].iloc[val_idx]
    fold_rows.append({
        'fold': fold_number,
        'train_rows': len(train_idx),
        'val_rows': len(val_idx),
        'val_positive_rate_%': round(val_target.mean() * 100, 2),
    })
fold_summary = pd.DataFrame(fold_rows)
print(f'Overall positive rate: {(train3_final[DATASET3.target_column].mean() * 100):.2f}%')
fold_summary

Number of folds: 5
Overall positive rate: 10.32%


,fold,train_rows,val_rows,val_positive_rate_%
0,0,147059,36765,10.32
1,1,147059,36765,10.32
2,2,147059,36765,10.32
3,3,147059,36765,10.32
4,4,147060,36764,10.32


**Result:** All 5 folds get essentially equal train (~4/5) / validation (~1/5) sizes, and each fold's validation positive rate stays close to the overall positive rate above -- confirming `stratified_kfold_splits()` preserves the class balance across folds, not just across the original train/test split. `src/main.py` runs this same call and writes the resulting fold labels to `DATASET3.kfold_indices_path`.

## Summary

| Dataset | Train rows | Test rows | Final features (excl. target) |
|---|---|---|---|
| Dataset 1 | 353,653 | 88,414 | 131 |
| Dataset 2 | 734 | 184 | 18 |
| Dataset 3 | 183,824 | 45,957 | 22 |

Every step above calls into `src/data/loader.py`, `src/data/preprocessing.py`, and `src/data/split.py` — the same functions `src/main.py`'s `process_dataset()` uses to actually write out `data/processed/<dataset>/{train,test}.csv` for all three datasets. Running `python src/main.py` reproduces this notebook's numbers end-to-end and persists the processed CSVs used for modeling.